# Hexiamond photo recognition — board pose from line segments

Prototype of the board-locating stage for a photo of a physical hexiamond puzzle:

1. Normalize the photo to a fixed long side.
2. Detect line segments (LSD) separately in the R, G and B channels; keep segments above a length threshold.
3. Merge the channels and weight each segment by **length × strength** (edge contrast across it).
4. Fold angles into `[0, 180)`, shift by +30 and take mod 60 — the three lattice directions collapse into one peak.
5. Keep segments within ±2° of that peak, then look at the three directions **unfolded**, at 0.1° resolution, and take each one's mode.
6. Fit the projective transform board → image
   $H = \begin{bmatrix} a & b & t_x \\ c & d & t_y \\ g & h & 1 \end{bmatrix}$.
   The **3×2 part** `[[a, b], [c, d], [g, h]]` carries in-plane rotation, stretch and keystone (trapezoid) deformation; `tx, ty` are the translation.
7. Draw the std board through `H` onto the photo.
8. Collect line evidence for every internal grid edge, cut the board into 6-cell blocks, colour them from the photo.
9. Express the blocks as the app's saved-board JSON.

The board geometry is read from `data.ts` next to this notebook, using the same vertex formula as `triangleVertices` in `geometry.ts` (edge length 1, y pointing down). The last step writes the result in the save format of `app/views/hexiamond.vue`.

In [ ]:
import json
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

plt.rcParams["figure.dpi"] = 80

# Repo root: the nearest ancestor holding package.json, so the notebook runs from any cwd inside the repo.
REPO = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "package.json").exists())
IMAGE_PATH = REPO / "footages.local" / "2026-07-16 23.01.03.jpg"
DATA_TS = REPO / "app" / "hexiamond" / "data.ts"
SHAPE_ID = "std"

LONG_SIDE = 1024        # normalized long side, px
MIN_LENGTH = 0.03       # min segment length, fraction of the long side
BLUR = 1.0              # Gaussian sigma before LSD, 0 disables
STRENGTH_OFFSET = 2.0   # px from the segment to each contrast sample
MOD_SHIFT = 30          # added before mod 60 so an axis-aligned board peaks mid-range
PEAK_HALF_WIDTH = 2.0   # deg, window around the folded peak
PEAK_STEP = 0.1         # deg, fine histogram bins
FIT_CLIPS = (24, 16, 10, 6)  # px, outlier clip per refinement round, wide to tight
PRIOR_FILL = 0.7             # prior board size as a fraction of the frame; kept small so the outline shrinks to fit
PRIOR_SCALE_STD = 0.2        # prior std of the scale, fraction of the prior mean
PRIOR_CENTER_STD = 0.2       # prior std of the board centre, fraction of the short image side
OUTSIDE_WEIGHT = 20.0        # weight of the outside-segment fraction against the prior's negative log
OUTSIDE_MARGIN = 0.3         # cell edges: segment parts closer than this to the outline don't count as outside (the frame)
COARSE_STEPS = 80
COARSE_RATE = 0.15           # step (in prior std units) per unit of loss
COARSE_MAX_STEP = 0.5        # cap on a single step, prior std units
MAX_KEYSTONE = 1.3           # max near/far scale ratio over the board in the projective fit
EDGE_ANGLE_TOLERANCE = 3.0   # deg, evidence segment vs projected grid edge
EDGE_OFFSET_TOLERANCE = 0.16 # fraction of the projected edge length, perpendicular
EDGE_SAMPLES = 64            # coverage samples along each grid edge
EDGE_THRESHOLDS = (0.1, 0.3, 0.5, 0.7)  # confidence tiers, low to high
PIECE_CELLS = 6              # every hexiamond piece covers 6 triangles

LATTICE_HEIGHT = math.sqrt(3) / 2
# (name, OpenCV channel index, RGB plot colour)
CHANNELS = [("B", 0, (0.0, 0.5, 1.0)), ("G", 1, (0.0, 0.78, 0.0)), ("R", 2, (1.0, 0.0, 0.0))]

def show(image, title=None, size=8):
	plt.figure(figsize=(size, size * image.shape[0] / image.shape[1]))
	plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if image.ndim == 3 else image, cmap="gray")
	plt.axis("off")
	if title:
		plt.title(title)
	plt.show()

## 1. Normalize resolution

In [ ]:
source = cv2.imread(str(IMAGE_PATH))
assert source is not None, f"cannot read {IMAGE_PATH}"
scale = LONG_SIDE / max(source.shape[:2])
image = cv2.resize(source, (round(source.shape[1] * scale), round(source.shape[0] * scale)), interpolation=cv2.INTER_AREA)
min_length = MIN_LENGTH * LONG_SIDE
print(f"source {source.shape[1]}x{source.shape[0]} -> {image.shape[1]}x{image.shape[0]} (scale {scale:.4f}), min length {min_length:.1f}px")
show(image, size=5)

## 2. Per-channel line segments

LSD runs on each colour channel on its own: piece borders that are faint in luminance often have strong contrast in one channel.
**Strength** is the mean `|I(+n) − I(−n)| / 255` sampled along the segment, `n` being its unit normal × 2px, in the channel that found it.

In [ ]:
def detect(channel):
	if BLUR > 0:
		channel = cv2.GaussianBlur(channel, (0, 0), BLUR)
	lines = cv2.createLineSegmentDetector(cv2.LSD_REFINE_STD).detect(channel)[0]
	return np.zeros((0, 4), np.float32) if lines is None else lines.reshape(-1, 4)

def lengths(lines):
	return np.hypot(lines[:, 2] - lines[:, 0], lines[:, 3] - lines[:, 1])

def strengths(channel, lines):
	channel = channel.astype(np.float32)
	result = np.zeros(len(lines), np.float32)
	for index, (x1, y1, x2, y2) in enumerate(lines):
		length = np.hypot(x2 - x1, y2 - y1)
		t = np.linspace(0, 1, max(int(length), 2), dtype=np.float32)
		xs, ys = x1 + (x2 - x1) * t, y1 + (y2 - y1) * t
		nx, ny = -(y2 - y1) / length * STRENGTH_OFFSET, (x2 - x1) / length * STRENGTH_OFFSET
		sample = lambda dx, dy: cv2.remap(channel, (xs + dx).reshape(-1, 1), (ys + dy).reshape(-1, 1), cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE).ravel()
		result[index] = np.abs(sample(nx, ny) - sample(-nx, -ny)).mean() / 255
	return result

segments, strength, channel_of = [], [], []
for name, index, _ in CHANNELS:
	lines = detect(image[:, :, index])
	kept = lines[lengths(lines) >= min_length]
	segments.append(kept)
	strength.append(strengths(image[:, :, index], kept))
	channel_of += [name] * len(kept)
	print(f"{name}: {len(lines)} segments, {len(kept)} >= threshold, mean strength {strength[-1].mean():.3f}")
segments = np.vstack(segments)
strength = np.concatenate(strength)
channel_of = np.array(channel_of)
weights = lengths(segments) * strength
print(f"merged {len(segments)} segments")

In [ ]:
def plot_segments(ax, lines, channels, background):
	ax.imshow(background, cmap="gray", vmin=0, vmax=255)
	for name, _, color in CHANNELS:
		for x1, y1, x2, y2 in lines[channels == name]:
			ax.plot([x1, x2], [y1, y2], color=color, linewidth=1.2)
	ax.set_xlim(0, image.shape[1])
	ax.set_ylim(image.shape[0], 0)
	ax.axis("off")

gray = (cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) * 0.35).astype(np.uint8)
fig, axes = plt.subplots(1, 4, figsize=(20, 7))
for ax, (name, index, _) in zip(axes, CHANNELS):
	mask = channel_of == name
	plot_segments(ax, segments[mask], channel_of[mask], image[:, :, index] // 3)
	ax.set_title(f"{name}: {mask.sum()} segments")
plot_segments(axes[3], segments, channel_of, gray)
axes[3].set_title(f"merged: {len(segments)}")
plt.tight_layout()
plt.show()

## 3. Folded angle distribution

Angles use image coordinates (y down), folded into `[0, 180)`. Adding 30 before `mod 60` maps lattice directions near 0/60/120° to ~30°, so the peak doesn't straddle the 0/60 seam.
The peak is the weighted circular mean on the 60° circle (angles × 6 → unit vectors).

In [ ]:
angles = np.degrees(np.arctan2(segments[:, 3] - segments[:, 1], segments[:, 2] - segments[:, 0])) % 180
folded = (angles + MOD_SHIFT) % 60

def circular_mean60(values, w):
	vector = (w * np.exp(1j * np.radians(values * 6))).sum()
	return (np.degrees(np.angle(vector)) / 6) % 60, abs(vector) / w.sum()

def stacked_histogram(ax, values, w, channels, bins):
	bottom = np.zeros(len(bins) - 1)
	for name, _, color in CHANNELS:
		mask = channels == name
		counts, _ = np.histogram(values[mask], bins=bins, weights=w[mask])
		ax.bar(bins[:-1], counts, width=bins[1] - bins[0], align="edge", bottom=bottom, color=color, label=f"{name} ({mask.sum()})")
		bottom += counts
	ax.set_xlim(bins[0], bins[-1])
	ax.grid(axis="y", alpha=0.3)
	return bottom

peak, resultant = circular_mean60(folded, weights)
fig, ax = plt.subplots(figsize=(12, 4))
stacked_histogram(ax, folded, weights, channel_of, np.arange(61))
ax.axvline(peak, color="black", linestyle="--", label=f"circular mean {peak:.2f}° (R={resultant:.2f})")
ax.set_xlabel(f"(angle + {MOD_SHIFT}) mod 60, 1° bins")
ax.set_ylabel("Σ length × strength")
ax.legend()
plt.show()
print(f"peak {peak:.2f}° -> lattice directions " + ", ".join(f"{(peak - MOD_SHIFT + k * 60) % 180:.2f}°" for k in range(3)))

## 4. Peak window, unfolded per direction

Segments within ±2° of the folded peak are kept (this also drops the off-board objects). Each is assigned to its lattice direction and plotted on its true angle at 0.1° resolution; the **mode** of each panel is the direction estimate used below.

In [ ]:
center = round(peak, 1)
window = np.abs((folded - center + 30) % 60 - 30) <= PEAK_HALF_WIDTH
axes_deg = [(center - MOD_SHIFT + k * 60) % 180 for k in range(3)]
direction = (np.round((angles - axes_deg[0]) / 60) % 3).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
modes = []
for k, ax in enumerate(axes):
	mask = window & (direction == k)
	unwrapped = axes_deg[k] + (angles - axes_deg[k] + 90) % 180 - 90
	bins = axes_deg[k] - PEAK_HALF_WIDTH + PEAK_STEP * np.arange(round(2 * PEAK_HALF_WIDTH / PEAK_STEP) + 1)
	counts = stacked_histogram(ax, unwrapped[mask], weights[mask], channel_of[mask], bins)
	modes.append(bins[np.argmax(counts)] + PEAK_STEP / 2)
	mean = np.average(unwrapped[mask], weights=weights[mask])
	ax.axvline(modes[-1], color="black", label=f"mode {modes[-1] % 180:.2f}°")
	ax.axvline(mean, color="gray", linestyle="--", label=f"mean {mean % 180:.2f}°")
	ax.set_title(f"{axes_deg[k]:.1f}° ± {PEAK_HALF_WIDTH:g}°: {mask.sum()} segments")
	ax.set_xlabel("segment angle (deg)")
	ax.legend(fontsize=8)
axes[0].set_ylabel("Σ length × strength")
plt.show()

fig, ax = plt.subplots(figsize=(6, 8))
plot_segments(ax, segments[window], channel_of[window], gray)
ax.set_title(f"{window.sum()} / {len(segments)} segments in window")
plt.show()
print("modes: " + ", ".join(f"{mode % 180:.2f}°" for mode in modes))

## 5. Board geometry

In [ ]:
def load_shape(shape_id):
	# RAW_SHAPES in data.ts is a JSON array literal: slice it out by bracket matching.
	text = DATA_TS.read_text(encoding="utf-8")
	start = text.index("= [", text.index("export const RAW_SHAPES")) + 2
	depth = 0
	for end, char in enumerate(text[start:], start):
		depth += (char == "[") - (char == "]")
		if not depth:
			break
	return next(shape for shape in json.loads(text[start:end + 1]) if shape["id"] == shape_id)

def triangle_vertices(packed, y):
	# triangleVertices(point, 1, 0) from geometry.ts.
	x, parity = packed // 2, packed % 2
	shear = lambda a, b: (a - b / 2, b * LATTICE_HEIGHT)
	return [shear(x, y), shear(x + parity, y + 1 - parity), shear(x + 1, y + 1)]

shape = load_shape(SHAPE_ID)
# Cells in buildShape's boardPoints order (row-major over packed x), so cell index == board point index.
board_points = [(packed, y) for y, row in enumerate(shape["board"]) for packed, value in enumerate(row) if value > 0]
triangles = np.array([triangle_vertices(packed, y) for packed, y in board_points])

def edges_by_count(triangles, wanted):
	# Lattice edges shared by `wanted` triangles: 1 = board outline, 2 = internal grid edge.
	counts = {}
	for triangle in triangles:
		for a, b in zip(triangle, np.roll(triangle, -1, axis=0)):
			key = tuple(sorted((tuple(np.round(a, 6)), tuple(np.round(b, 6)))))
			counts[key] = counts.get(key, 0) + 1
	return np.array([key for key, count in counts.items() if count == wanted])

outline = edges_by_count(triangles, 1)
internal = edges_by_count(triangles, 2)
vertices = triangles.reshape(-1, 2)
print(f"{SHAPE_ID}: {len(triangles)} triangles, {len(outline)} outline edges, {len(internal)} internal edges")

## 6. Fit the transform

### 6.1 Linear part `[[a, b], [c, d]]` from the modes

Board lattice directions are $u_0 = 0°$, $u_1 = 60°$, $u_2 = 120°$ with $u_2 = u_1 - u_0$. Requiring $L u_k \parallel e_k$ (unit vectors at the three modes) gives
$L u_0 = e_0$, $L u_1 = \lambda_1 e_1$ and $\lambda_1 e_1 - \lambda_2 e_2 = e_0$ — a 2×2 solve. Three directions fix $L$ up to scale, so it is normalized to det 1.
`-L` sends every direction to the same line, so the sign that keeps the board upright is chosen.

Angles alone can't determine `g, h`: with perspective a direction no longer maps to one image angle, it fans out across the board. So `g = h = 0` here, and the keystone is fitted in 6.3 from where the segments actually lie.

In [ ]:
unit = lambda deg: np.array([math.cos(math.radians(deg)), math.sin(math.radians(deg))])

def linear_from_directions(image_angles):
	e0, e1, e2 = (unit(angle) for angle in image_angles)
	l1, _ = np.linalg.solve(np.column_stack([e1, -e2]), e0)
	linear = np.column_stack([e0, l1 * e1]) @ np.linalg.inv(np.column_stack([unit(0), unit(60)]))
	assert np.linalg.det(linear) > 0, "directions imply a mirrored board"
	if linear[0, 0] < 0:
		linear = -linear
	return linear / math.sqrt(np.linalg.det(linear))

def polar(linear):
	# L = R(rotation) S, S symmetric: report rotation and S's stretch ratio and major axis.
	u, sigma, vt = np.linalg.svd(linear)
	rotation = u @ vt
	return math.degrees(math.atan2(rotation[1, 0], rotation[0, 0])), sigma[0] / sigma[1], math.degrees(math.atan2(vt[0, 1], vt[0, 0])) % 180

linear = linear_from_directions(modes)
rotation, stretch, stretch_axis = polar(linear)
print(f"L (det 1) =\n{linear.round(5)}\nrotation {rotation:.2f}°, stretch {stretch:.4f} along {stretch_axis:.1f}°")

### 6.2 Scale and translation (affine)

With the linear part fixed, the board sits at $x = s L p + t$. It is parametrized by $(s, c_x, c_y)$ — the scale and the image position of the board's centroid — and located in two stages.

**Coarse: prior + segments outside the board.** The loss is

$$\mathcal{L}(s, c) = \tfrac12 \Big\| \tfrac{(s, c_x, c_y) - \mu}{\sigma} \Big\|^2 + \lambda \frac{\sum_i \mathrm{strength}_i \cdot \mathrm{outside}_i}{\sum_i \mathrm{strength}_i \cdot \mathrm{length}_i}$$

* the first term is the negative log of a Gaussian prior: the centroid near the image centre, and the scale at a board filling `PRIOR_FILL` (0.7) of the frame — on purpose a bit smaller than photos usually show, so the prior pulls the outline *inward* and the segments push it out just until they are inside;
* the second is the strength-weighted length of the window segments falling outside the projected board outline grown by `OUTSIDE_MARGIN` (0.3 cell), as a fraction of the total. Each segment is clipped against that convex polygon. The margin matters: the board's dark frame gives long, strong lines just beyond the outline, and without it a third of the weight sits "outside" even at the right pose, which drowns the signal.

Descent starts at $\mu$ and works in σ units: a central-difference gradient gives the direction, and the step length is `COARSE_RATE × loss` (capped at `COARSE_MAX_STEP` σ) — large while the loss is large, shrinking as it settles. After `COARSE_STEPS` steps the lowest-loss point is the coarse pose.

**Fine: lattice alignment.** Lattice lines satisfy $n_k \cdot p \in h\mathbb{Z}$ in board space ($h = \sqrt3/2$). Each window segment midpoint is mapped back to the board, its distance to the nearest line of its own direction (in px) is the residual, and $\sum w \min(r^2, c^2)$ is minimized: a grid over ±half a cell in translation and ±10 % in scale around the coarse pose, then Nelder–Mead.

**Cell choice.** The lattice term is periodic — a board shifted by a whole lattice vector fits the interior lines exactly as well — so after the fine fit the pose walks to whichever of its six one-cell neighbours has a lower coarse loss (prior + outside), until none does. Mixing the outside term into the fine objective instead doesn't work: without the prior it simply rewards a bigger board. On the first sample, with the margin, the outside share is 0.000 at the right pose and ≥ 0.17 at every one-cell neighbour.

In [ ]:
mids = (segments[:, :2] + segments[:, 2:]) / 2
board_normals = np.array([unit(k * 60 + 90) for k in range(3)])

def to_board(H, points):
	q = np.c_[points, np.ones(len(points))] @ np.linalg.inv(H).T
	return q[:, :2] / q[:, 2:]

def to_image(H, points):
	q = np.c_[points, np.ones(len(points))] @ H.T
	return q[:, :2] / q[:, 2:]

def residuals(H, points, dirs, px_per_unit):
	offset = np.einsum("ij,ij->i", to_board(H, points), board_normals[dirs]) / LATTICE_HEIGHT
	return (offset - np.round(offset)) * LATTICE_HEIGHT * px_per_unit

def clipped_loss(H, points, dirs, w, clip, px_per_unit):
	r = residuals(H, points, dirs, px_per_unit)
	return (w * np.minimum(r ** 2, clip ** 2)).sum() / w.sum()

def inlier_share(H, points, dirs, w, px_per_unit, clip=6):
	r = np.abs(residuals(H, points, dirs, px_per_unit))
	inliers = r <= clip
	return w[inliers].sum() / w.sum(), math.sqrt(np.average(r[inliers] ** 2, weights=w[inliers])) if inliers.any() else float("nan")

def report(label, H, points, dirs, w, px_per_unit, clip=6):
	share, rms = inlier_share(H, points, dirs, w, px_per_unit, clip)
	print(f"{label}: inliers (<= {clip}px) {share:.1%} of weight, inlier rms {rms:.2f}px")

fit_segments, fit_dirs, fit_w = segments[window], direction[window], weights[window]
fit_mids, fit_strength, fit_lengths = mids[window], strength[window], lengths(segments[window])

# Board outline as a convex polygon (the std board is a truncated triangle), and its centroid.
hull = cv2.convexHull(vertices.astype(np.float32)).reshape(-1, 2).astype(np.float64)
centroid_board = vertices.mean(axis=0)

def offset_polygon(polygon, margin):
	# Push every edge of a convex polygon outward by `margin` and re-intersect neighbouring edges.
	orientation = np.sign(np.cross(polygon[1] - polygon[0], polygon[2] - polygon[0]))
	lines = []
	for p0, p1 in zip(polygon, np.roll(polygon, -1, axis=0)):
		e = (p1 - p0) / np.linalg.norm(p1 - p0)
		outward = -np.array([-e[1], e[0]]) * orientation
		lines.append((outward, outward @ p0 + margin))
	return np.array([np.linalg.solve(np.array([lines[i - 1][0], lines[i][0]]), np.array([lines[i - 1][1], lines[i][1]])) for i in range(len(lines))])

margin_hull = offset_polygon(hull, OUTSIDE_MARGIN)

def pose_H(params):
	scale_px, cx, cy = params
	t = np.array([cx, cy]) - scale_px * linear @ centroid_board
	return np.vstack([np.column_stack([scale_px * linear, t]), [0, 0, 1]])

def outside_lengths(polygon):
	# Cyrus-Beck clip of every window segment against the convex polygon, vectorized over segments.
	start, delta = fit_segments[:, :2], fit_segments[:, 2:] - fit_segments[:, :2]
	lo, hi = np.zeros(len(start)), np.ones(len(start))
	orientation = np.sign(np.cross(polygon[1] - polygon[0], polygon[2] - polygon[0]))
	for p0, p1 in zip(polygon, np.roll(polygon, -1, axis=0)):
		inward = np.array([-(p1 - p0)[1], (p1 - p0)[0]]) * orientation
		num, den = (start - p0) @ inward, delta @ inward
		with np.errstate(divide="ignore", invalid="ignore"):
			t = -num / den
		lo = np.where(den > 0, np.maximum(lo, t), lo)
		hi = np.where(den < 0, np.minimum(hi, t), hi)
		hi = np.where((den == 0) & (num < 0), -1.0, hi)
	return fit_lengths * (1 - np.clip(hi - lo, 0, 1))

extent = np.ptp(vertices @ linear.T, axis=0)
prior_mean = np.array([PRIOR_FILL * min(image.shape[1] / extent[0], image.shape[0] / extent[1]), image.shape[1] / 2, image.shape[0] / 2])
prior_std = np.array([PRIOR_SCALE_STD * prior_mean[0], PRIOR_CENTER_STD * min(image.shape[:2]), PRIOR_CENTER_STD * min(image.shape[:2])])

def coarse_loss(params):
	prior = 0.5 * (((params - prior_mean) / prior_std) ** 2).sum()
	polygon = to_image(pose_H(params), margin_hull)
	outside = (fit_strength * outside_lengths(polygon)).sum() / (fit_strength * fit_lengths).sum()
	return prior + OUTSIDE_WEIGHT * outside, prior, outside

z = np.zeros(3)  # offset from the prior mean, in prior std units
coarse_path = [prior_mean.copy()]
best = (coarse_loss(prior_mean)[0], prior_mean.copy())
for _ in range(COARSE_STEPS):
	value = coarse_loss(prior_mean + z * prior_std)[0]
	gradient = np.array([(coarse_loss(prior_mean + (z + h) * prior_std)[0] - coarse_loss(prior_mean + (z - h) * prior_std)[0]) / 0.02 for h in np.eye(3) * 0.01])
	norm = np.linalg.norm(gradient)
	if norm < 1e-9:
		break
	z = z - min(COARSE_MAX_STEP, COARSE_RATE * value) * gradient / norm
	params = prior_mean + z * prior_std
	coarse_path.append(params)
	if coarse_loss(params)[0] < best[0]:
		best = (coarse_loss(params)[0], params)
coarse = best[1]
total, prior_part, outside_part = coarse_loss(coarse)
print(f"prior mean s {prior_mean[0]:.1f}, c ({prior_mean[1]:.0f}, {prior_mean[2]:.0f}); std {prior_std.round(1)}")
print(f"coarse: s {coarse[0]:.2f} px per edge, c ({coarse[1]:.1f}, {coarse[2]:.1f}); loss {total:.3f} = prior {prior_part:.3f} + {OUTSIDE_WEIGHT:g} x outside {outside_part:.3f}")

half_cell = 0.5 * coarse[0]
candidates = [(coarse[0] * f, coarse[1] + dx, coarse[2] + dy) for f in np.linspace(0.9, 1.1, 11) for dx in np.linspace(-half_cell, half_cell, 11) for dy in np.linspace(-half_cell, half_cell, 11)]
fine_loss = lambda p: clipped_loss(pose_H(p), fit_mids, fit_dirs, fit_w, FIT_CLIPS[-1], p[0])
start = min(candidates, key=fine_loss)
result = minimize(fine_loss, start, method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-6, "maxiter": 4000})

# Whole-cell shifts fit the lattice equally well; walk to the neighbour with the lowest coarse loss (prior + outside) until none is better.
lattice_steps = [np.array([math.cos(math.radians(a)), math.sin(math.radians(a))]) for a in range(0, 360, 60)]
pose = result.x.copy()
for _ in range(8):
	shifted = [pose] + [np.array([pose[0], *(pose[1:] + pose[0] * linear @ step)]) for step in lattice_steps]
	best_shift = min(shifted, key=lambda p: coarse_loss(p)[0])
	if best_shift is pose:
		break
	print(f"shift by one cell: c ({pose[1]:.0f}, {pose[2]:.0f}) -> ({best_shift[1]:.0f}, {best_shift[2]:.0f}), coarse loss {coarse_loss(pose)[0]:.3f} -> {coarse_loss(best_shift)[0]:.3f}")
	pose = best_shift
result.x = pose
H_affine = pose_H(result.x)
px_per_unit = result.x[0]
print(f"affine: s {px_per_unit:.2f} px per edge, t ({H_affine[0, 2]:.2f}, {H_affine[1, 2]:.2f})")
report("affine", H_affine, fit_mids, fit_dirs, fit_w, px_per_unit)

fig, ax = plt.subplots(figsize=(6, 8))
plot_segments(ax, fit_segments, channel_of[window], gray)
for params, color, label in ((prior_mean, "magenta", "prior mean"), (coarse, "yellow", "coarse"), (result.x, "cyan", "fine (affine)")):
	polygon = to_image(pose_H(params), hull)
	ax.plot(*np.vstack([polygon, polygon[:1]]).T, color=color, linewidth=2, label=label)
path = np.array(coarse_path)
ax.plot(path[:, 1], path[:, 2], ".-", color="yellow", markersize=3, linewidth=0.8)
ax.legend(loc="lower right", fontsize=8)
ax.set_title("board outline: prior mean -> coarse -> fine")
plt.show()

### 6.3 Keystone `g, h` (full projective refinement)

Starting from the affine fit, all 8 parameters are refined against both endpoints of every window segment — two points per segment also pin its local angle, which is what perspective changes.
The outlier clip starts wide and tightens (`FIT_CLIPS`): with a tight clip from the start, the rows that the affine fit misses by more than the clip (the near, bottom rows) count as outliers and never pull the fit.

Two guards keep a poor start from running away: the perspective denominator $w = g x + h y + 1$ over the board must stay positive with a near/far ratio under `MAX_KEYSTONE` (otherwise the loss is infinite), and if the projective fit ends with a smaller inlier share than the affine one, the affine fit is kept.

In [ ]:
ends = np.vstack([segments[window][:, :2], segments[window][:, 2:]])
end_dirs = np.concatenate([fit_dirs, fit_dirs])
end_w = np.concatenate([fit_w, fit_w])

x = H_affine.ravel()[:8].copy()
x_scale = np.abs(x) + np.array([0, 0, 0, 0, 0, 0, 1e-4, 1e-4])
H_of = lambda vector: np.append(vector, 1).reshape(3, 3)

def bounded_loss(vector, clip):
	w = vertices @ vector[6:8] + 1
	if w.min() <= 0 or w.max() / w.min() > MAX_KEYSTONE:
		return np.inf
	return clipped_loss(H_of(vector), ends, end_dirs, end_w, clip, px_per_unit)

report("affine (endpoints)", H_affine, ends, end_dirs, end_w, px_per_unit)
for clip in FIT_CLIPS:
	step = minimize(lambda y: bounded_loss(x + y * x_scale, clip), np.zeros(8), method="Powell", options={"maxiter": 20000, "xtol": 1e-5, "ftol": 1e-9})
	x = x + step.x * x_scale
	report(f"projective, clip {clip:>2}px", H_of(x), ends, end_dirs, end_w, px_per_unit)
H = H_of(x)
if inlier_share(H, ends, end_dirs, end_w, px_per_unit)[0] < inlier_share(H_affine, ends, end_dirs, end_w, px_per_unit)[0]:
	print("projective fit is worse than affine -> keeping the affine fit")
	H = H_affine

### 6.4 The 3×2 part

`[[a, b], [c, d], [g, h]]` — everything but the translation. Its effect on the board:

* **rotation / stretch**: polar decomposition of `[[a, b], [c, d]]`;
* **keystone**: the denominator $w = g x + h y + 1$ over the board — local scale goes as $1/w$, so its range is the near/far size ratio, and $(g, h)$ points toward the far side;
* the three edge directions of an equilateral cell, mapped at the top, centre and bottom of the board, compared with the modes.

In [ ]:
part = np.array([[H[0, 0], H[0, 1]], [H[1, 0], H[1, 1]], [H[2, 0], H[2, 1]]])
np.set_printoptions(suppress=True, precision=6)
print(f"3x2 part [[a, b], [c, d], [g, h]] =\n{part}\ntranslation (tx, ty) = ({H[0, 2]:.3f}, {H[1, 2]:.3f})   [normalized px]")
rotation, stretch, stretch_axis = polar(part[:2] / math.sqrt(np.linalg.det(part[:2])))
w = vertices @ part[2] + 1
far = math.degrees(math.atan2(part[2, 1], part[2, 0])) % 360
print(f"rotation {rotation:.2f}°, stretch {stretch:.4f} along {stretch_axis:.1f}°")
print(f"keystone: w over the board {w.min():.4f} .. {w.max():.4f} -> near/far scale ratio {w.max() / w.min():.4f}, far side toward board direction {far:.1f}°")

def edge_angles(p):
	# Image angles of the three lattice directions through board point p.
	base = to_image(H, np.array([p]))[0]
	result = []
	for k in range(3):
		delta = to_image(H, np.array([p + 0.01 * unit(k * 60)]))[0] - base
		result.append(math.degrees(math.atan2(delta[1], delta[0])) % 180)
	return result

centroid = vertices.mean(axis=0)
for label, y in (("top", vertices[:, 1].min() + 0.5), ("centre", centroid[1]), ("bottom", vertices[:, 1].max() - 0.5)):
	print(f"{label:>6} (board y {y:.2f}): " + ", ".join(f"{angle:.2f}°" for angle in edge_angles(np.array([centroid[0], y]))))
print(f" modes: " + ", ".join(f"{mode % 180:.2f}°" for mode in modes))
source_H = np.diag([1 / scale, 1 / scale, 1]) @ H
print(f"\nH in source px (x 1/scale):\n{source_H}")

## 7. Board overlay

In [ ]:
def draw_board(H, label):
	canvas = image.copy()
	for triangle in triangles:
		cv2.polylines(canvas, [np.round(to_image(H, triangle) * 16).astype(np.int32)], True, (255, 255, 0), 1, cv2.LINE_AA, shift=4)
	for a, b in to_image(H, outline.reshape(-1, 2)).reshape(-1, 2, 2):
		cv2.line(canvas, tuple(np.round(a * 16).astype(int)), tuple(np.round(b * 16).astype(int)), (0, 255, 255), 2, cv2.LINE_AA, shift=4)
	cv2.putText(canvas, label, (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
	return canvas

fig, axes = plt.subplots(1, 2, figsize=(16, 11))
for ax, (matrix, label) in zip(axes, ((H_affine, "affine (g = h = 0)"), (H, "projective"))):
	ax.imshow(cv2.cvtColor(draw_board(matrix, label), cv2.COLOR_BGR2RGB))
	ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Edge evidence per internal grid edge

Each of the board's internal lattice edges (shared by two cells) is projected through `H`. A detected segment (any of the merged ones, not just the ±2° window) counts as evidence for it when

* its angle is within ±3° of the projected edge's angle, and
* clipped to the edge's extent, both clipped ends lie within 0.16 × the projected edge length of the edge line — the edge length is the local cell size, so the tolerance follows the perspective.

**Confidence**: the edge is sampled at 64 points; each sample takes the highest strength among evidence pieces covering it, divided by the 90th-percentile segment strength and capped at 1; confidence is the mean over samples. So it is coverage × contrast: a fully covered, high-contrast edge scores 1; a piece interior, crossed by no border, scores ~0. Overlapping detections (three channels, rim + gap + shadow) don't add up, the max wins.

Each panel shows the evidence pieces of edges at or above one threshold, opacity = that edge's confidence; grey lines are all internal edges.

In [ ]:
strength_ref = np.percentile(strength, 90)

def edge_evidence(a, b):
	length = np.linalg.norm(b - a)
	u = (b - a) / length
	n = np.array([-u[1], u[0]])
	theta = math.degrees(math.atan2(u[1], u[0])) % 180
	near = np.abs((angles - theta + 90) % 180 - 90) <= EDGE_ANGLE_TOLERANCE
	level = np.zeros(EDGE_SAMPLES)
	samples = (np.arange(EDGE_SAMPLES) + 0.5) / EDGE_SAMPLES
	pieces = []
	for index in np.flatnonzero(near):
		p1, p2 = segments[index, :2] - a, segments[index, 2:] - a
		t1, t2 = p1 @ u / length, p2 @ u / length
		o1, o2 = p1 @ n, p2 @ n
		if abs(t2 - t1) < 1e-9:
			continue
		lo, hi = max(0.0, min(t1, t2)), min(1.0, max(t1, t2))
		if hi <= lo:
			continue
		offset = lambda t: o1 + (o2 - o1) * (t - t1) / (t2 - t1)
		if max(abs(offset(lo)), abs(offset(hi))) > EDGE_OFFSET_TOLERANCE * length:
			continue
		covered = (samples >= lo) & (samples <= hi)
		level[covered] = np.maximum(level[covered], min(1.0, strength[index] / strength_ref))
		pieces.append([a + u * length * t + n * offset(t) for t in (lo, hi)])
	return level.mean(), np.array(pieces).reshape(-1, 2, 2)

projected_edges = to_image(H, internal.reshape(-1, 2)).reshape(-1, 2, 2)
evidence = [edge_evidence(a, b) for a, b in projected_edges]
confidence = np.array([value for value, _ in evidence])
print(f"strength reference (p90) {strength_ref:.3f}; {len(internal)} internal edges, {sum(len(p) for _, p in evidence)} evidence pieces")
print("confidence quantiles 0/25/50/75/100%: " + ", ".join(f"{q:.2f}" for q in np.percentile(confidence, [0, 25, 50, 75, 100])))
for threshold in EDGE_THRESHOLDS:
	print(f">= {threshold:.1f}: {(confidence >= threshold).sum()} edges")

In [ ]:
from matplotlib.collections import LineCollection

background = cv2.cvtColor((cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) * 0.45).astype(np.uint8), cv2.COLOR_GRAY2RGB)
fig, axes = plt.subplots(2, 2, figsize=(16, 21))
for ax, threshold in zip(axes.ravel(), EDGE_THRESHOLDS):
	ax.imshow(background)
	ax.add_collection(LineCollection(projected_edges, colors=[(0.6, 0.6, 0.6, 0.35)], linewidths=0.8))
	lines, colors = [], []
	for (value, pieces) in evidence:
		if value >= threshold:
			lines += list(pieces)
			colors += [(1.0, 0.85, 0.0, value)] * len(pieces)
	ax.add_collection(LineCollection(lines, colors=colors, linewidths=2.2))
	ax.set_title(f"confidence >= {threshold:.1f}: {(confidence >= threshold).sum()} / {len(internal)} edges")
	ax.set_xlim(0, image.shape[1])
	ax.set_ylim(image.shape[0], 0)
	ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Partition by cutting evidence edges, high confidence first

Internal edges are marked as cuts tier by tier, from the highest threshold down; a tier's batch is the edges whose confidence falls in `[threshold, previous threshold)`. Cells joined through uncut internal edges form a region.

After each batch, any region smaller than 6 cells can't hold a piece, so the batch over-cut there: the edges **of this batch** touching a cell of such a region are rejected, and the check repeats until every region has ≥ 6 cells. Earlier batches already passed the check, so dropping the new edges can only merge regions back toward that state — the loop terminates.

Regions larger than 6 (under-partition) are left alone in this step.

In [ ]:
def cell_key(point):
	return tuple(np.round(point, 6))

edge_cells = {}
for cell, triangle in enumerate(triangles):
	for a, b in zip(triangle, np.roll(triangle, -1, axis=0)):
		edge_cells.setdefault(tuple(sorted((cell_key(a), cell_key(b)))), []).append(cell)
internal_pairs = np.array([edge_cells[tuple(map(tuple, edge))] for edge in internal])

def regions(cut):
	# Union-find over cells joined by uncut internal edges; returns a region label per cell.
	parent = list(range(len(triangles)))
	def find(i):
		while parent[i] != i:
			parent[i] = parent[parent[i]]
			i = parent[i]
		return i
	for (a, b), is_cut in zip(internal_pairs, cut):
		if not is_cut:
			parent[find(a)] = find(b)
	roots = [find(i) for i in range(len(triangles))]
	_, labels = np.unique(roots, return_inverse=True)
	return labels

cut = np.zeros(len(internal), bool)
rejected = np.zeros(len(internal), bool)
stages = []
upper = np.inf
for threshold in sorted(EDGE_THRESHOLDS, reverse=True):
	batch = (confidence >= threshold) & (confidence < upper)
	upper = threshold
	trial = cut | batch
	rounds = 0
	while True:
		labels = regions(trial)
		sizes = np.bincount(labels)
		small_cells = np.flatnonzero(sizes[labels] < PIECE_CELLS)
		if not len(small_cells):
			break
		touching = np.isin(internal_pairs, small_cells).any(axis=1) & batch & trial
		trial &= ~touching
		rejected |= touching
		rounds += 1
	accepted = trial & batch
	cut = trial
	sizes = np.bincount(regions(cut))
	stages.append((threshold, batch.copy(), accepted.copy(), cut.copy(), rejected.copy()))
	print(f">= {threshold:.1f}: batch {batch.sum():>2}, accepted {accepted.sum():>2}, rejected {(batch & ~accepted).sum():>2} ({rounds} rounds) -> "
		f"{len(sizes)} regions, sizes {sorted(sizes.tolist())}")
labels = regions(cut)
sizes = np.bincount(labels)
print(f"final: {cut.sum()} cut edges, {len(sizes)} regions; exactly 6 cells: {(sizes == PIECE_CELLS).sum()}, larger: {(sizes > PIECE_CELLS).sum()} "
	f"(cells in larger regions {sizes[sizes > PIECE_CELLS].sum()})")

In [ ]:
from matplotlib.patches import Polygon

def plot_partition(ax, cut, rejected_edges, title):
	labels = regions(cut)
	sizes = np.bincount(labels)
	ax.imshow(background)
	palette = plt.get_cmap("tab20")
	for cell, triangle in enumerate(triangles):
		region = labels[cell]
		color = palette(region % 20) if sizes[region] == PIECE_CELLS else (1.0, 1.0, 1.0)
		ax.add_patch(Polygon(to_image(H, triangle), closed=True, facecolor=color, alpha=0.45 if sizes[region] == PIECE_CELLS else 0.15, edgecolor="none"))
	ax.add_collection(LineCollection(projected_edges[~cut], colors=[(0.7, 0.7, 0.7, 0.4)], linewidths=0.6))
	ax.add_collection(LineCollection(projected_edges[cut], colors=[(1.0, 0.85, 0.0, 1.0)], linewidths=2.4))
	ax.add_collection(LineCollection(projected_edges[rejected_edges], colors=[(1.0, 0.1, 0.1, 0.9)], linewidths=1.6, linestyles="--"))
	ax.add_collection(LineCollection(to_image(H, outline.reshape(-1, 2)).reshape(-1, 2, 2), colors=[(1.0, 0.85, 0.0, 1.0)], linewidths=2.4))
	for region in range(len(sizes)):
		center = to_image(H, triangles[labels == region].reshape(-1, 2).mean(axis=0, keepdims=True))[0]
		ax.text(*center, str(sizes[region]), color="white", fontsize=11, ha="center", va="center", weight="bold")
	ax.set_title(title)
	ax.set_xlim(0, image.shape[1])
	ax.set_ylim(image.shape[0], 0)
	ax.axis("off")

fig, axes = plt.subplots(2, 2, figsize=(16, 21))
for ax, (threshold, batch, accepted, stage_cut, stage_rejected) in zip(axes.ravel(), stages):
	plot_partition(ax, stage_cut, stage_rejected & batch, f"after >= {threshold:.1f}: +{accepted.sum()} cut, {(batch & ~accepted).sum()} rejected (red dashed)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 13))
plot_partition(ax, cut, rejected, f"after all tiers: {len(sizes)} regions, {(sizes == PIECE_CELLS).sum()} of 6 cells (coloured); numbers = cells per region")
plt.show()

## 10. Under-partition: colour-guided split

Regions larger than 6 cells are resolved with the photo's colours.

**Per-cell colour.** Every board cell is rasterized in the full-resolution photo (through `H / scale`), eroded by 0.08 edge, and summarized as a saturation-weighted circular-mean **hue** and a mean **brightness** (HSV V).

**Homogeneity.** For a set of cells, each cell's deviation is measured from the set's median — circular for hue (the member hue closest to all others), plain for brightness. The set is *uniform* when every cell is within both thresholds.

**Thresholds** come from this image's own partition: the 6-cell regions left by §9 (the confidently cut blocks) give per-cell deviations from their block median; the 95th percentile of those is the threshold, for hue and brightness separately. A percentile rather than the max: a single off cell (glare, a stud) would otherwise set the bar.

**Per region > 6 cells** (a queue, worked until no region > 6 is left)

1. Uniform as a whole → stop; the region is accepted as **blank** (no block): one colour over more than a piece's area is bare board.
2. Otherwise look for a uniform connected 6-cell set inside it (connected through uncut edges). Candidates that leave no remainder fragment under 6 cells are preferred, then the smallest normalized deviation `max(hue_dev / T_hue, value_dev / T_value)`.
   * Found → carve it: the added cuts are exactly the edges between the set and the rest of the region, so no stray line ends inside a piece. Remainder components > 6 go back into the queue; a component of exactly 6 stays as a region; a fragment under 6 can't be a piece and becomes blank.
   * Not found → stop; the whole region is accepted as blank.

The sample needs none of this after §9, so the cell also runs it on the earlier stages for illustration.

In [ ]:
source_H = np.diag([1 / scale, 1 / scale, 1]) @ H
source_hsv = cv2.cvtColor(source, cv2.COLOR_BGR2HSV)
edge_px = np.mean([np.linalg.norm(b - a) for a, b in to_image(source_H, internal.reshape(-1, 2)).reshape(-1, 2, 2)])

def circular_mean_hue(pixels):
	angle = np.radians(pixels[:, 0] * 2)
	weight = pixels[:, 1] + 1e-6
	return np.degrees(np.arctan2((weight * np.sin(angle)).sum(), (weight * np.cos(angle)).sum())) % 360

cell_erode = np.ones((2 * round(0.08 * edge_px) + 1,) * 2, np.uint8)
cell_hue, cell_value = np.zeros(len(triangles)), np.zeros(len(triangles))
for cell, triangle in enumerate(triangles):
	mask = np.zeros(source.shape[:2], np.uint8)
	cv2.fillPoly(mask, [np.round(to_image(source_H, triangle) * 16).astype(np.int32)], 255, cv2.LINE_8, shift=4)
	pixels = source_hsv[cv2.erode(mask, cell_erode) > 0].astype(np.float64)
	# A cell projected outside the photo has no pixels: NaN, which spread() treats as never uniform.
	cell_hue[cell] = circular_mean_hue(pixels) if len(pixels) else np.nan
	cell_value[cell] = pixels[:, 2].mean() if len(pixels) else np.nan

def deviations(cells):
	cells = list(cells)
	hue, value = cell_hue[cells], cell_value[cells]
	distance = np.abs((hue[:, None] - hue[None, :] + 180) % 360 - 180)
	center = hue[np.argmin(distance.sum(axis=1))]
	return np.abs((hue - center + 180) % 360 - 180), np.abs(value - np.median(value))

labels = regions(cut)
sizes = np.bincount(labels)
reference = [deviations(np.flatnonzero(labels == region)) for region in np.flatnonzero(sizes == PIECE_CELLS)]
if reference:
	HUE_THRESHOLD = np.percentile(np.concatenate([hue for hue, _ in reference]), 95)
	VALUE_THRESHOLD = np.percentile(np.concatenate([value for _, value in reference]), 95)
	print(f"thresholds from {len(reference)} six-cell blocks (p95 of per-cell deviation): hue {HUE_THRESHOLD:.1f}°, brightness {VALUE_THRESHOLD:.1f}")
else:
	HUE_THRESHOLD = VALUE_THRESHOLD = None
	print("no six-cell block after §9: colour thresholds unavailable, under-partitioned regions stay as they are")

def spread(cells):
	hue, value = deviations(cells)
	if np.isnan(hue).any() or np.isnan(value).any():
		return np.inf
	return max(hue.max() / HUE_THRESHOLD, value.max() / VALUE_THRESHOLD)

def uniform(cells):
	return spread(cells) <= 1

def components(cells, neighbors):
	cells, result = set(cells), []
	while cells:
		stack, part = [cells.pop()], set()
		while stack:
			cell = stack.pop()
			part.add(cell)
			for n in neighbors[cell]:
				if n in cells:
					cells.remove(n)
					stack.append(n)
		result.append(part)
	return result

def uncut_neighbors(cut):
	neighbors = [[] for _ in triangles]
	for (a, b), is_cut in zip(internal_pairs, cut):
		if not is_cut:
			neighbors[a].append(b)
			neighbors[b].append(a)
	return neighbors

def six_sets(cells, neighbors, size=PIECE_CELLS):
	# Every connected `size`-cell subset of `cells`, grown by frontier extension, deduplicated across seeds.
	seen, found = set(), []
	stack = [frozenset([cell]) for cell in cells]
	while stack:
		current = stack.pop()
		if current in seen:
			continue
		seen.add(current)
		if len(current) == size:
			found.append(current)
			continue
		for n in {n for c in current for n in neighbors[c] if n in cells and n not in current}:
			stack.append(current | {n})
	return found

def resolve_under_partition(cut):
	cut = cut.copy()
	blank = np.zeros(len(triangles), bool)
	notes = []
	labels = regions(cut)
	queue = [set(np.flatnonzero(labels == region)) for region in np.flatnonzero(np.bincount(labels) > PIECE_CELLS)]
	while queue:
		cells = queue.pop()
		if uniform(cells):
			blank[list(cells)] = True
			notes.append(f"{len(cells)} cells uniform -> blank")
			continue
		neighbors = uncut_neighbors(cut)
		best = None
		for part in six_sets(cells, neighbors):
			score = spread(part)
			if score > 1:
				continue
			rest = components(cells - part, neighbors)
			key = (any(len(rest_part) < PIECE_CELLS for rest_part in rest), score)
			if best is None or key < best[0]:
				best = (key, part, rest)
		if best is None:
			blank[list(cells)] = True
			notes.append(f"{len(cells)} cells: no uniform 6-set -> blank")
			continue
		(_, score), part, rest = best
		added = [i for i, (a, b) in enumerate(internal_pairs) if a in cells and b in cells and (a in part) != (b in part) and not cut[i]]
		cut[added] = True
		notes.append(f"{len(cells)} cells: carved a uniform 6-set (spread {score:.2f}), +{len(added)} cuts, rest {sorted(len(r) for r in rest)}")
		for rest_part in rest:
			if len(rest_part) > PIECE_CELLS:
				queue.append(rest_part)
			elif len(rest_part) < PIECE_CELLS:
				blank[list(rest_part)] = True
	return cut, blank, notes

if reference:
	final_cut, blank_cells, notes = resolve_under_partition(cut)
else:
	final_cut, blank_cells, notes = cut.copy(), np.zeros(len(triangles), bool), ["skipped, no thresholds"]
print("sample:", "; ".join(notes) or "no region larger than 6, nothing to split")

demo = [stage for stage in stages if reference and (np.bincount(regions(stage[3])) > PIECE_CELLS).any()]
fig, axes = plt.subplots(max(len(demo), 1), 2, figsize=(14, 9.5 * max(len(demo), 1)))
for row, (threshold, _, _, stage_cut, _) in zip(np.atleast_2d(axes), demo):
	split, stage_blank, stage_notes = resolve_under_partition(stage_cut)
	print(f"\ndemo after >= {threshold:.1f}:\n  " + "\n  ".join(stage_notes))
	plot_partition(row[0], stage_cut, np.zeros(len(internal), bool), f"after >= {threshold:.1f}")
	plot_partition(row[1], split, split & ~stage_cut, "+ colour-guided split (red dashed = added)")
	for cell in np.flatnonzero(stage_blank):
		row[1].add_patch(Polygon(to_image(H, triangles[cell]), closed=True, facecolor="none", edgecolor="white", hatch="//", linewidth=0))
plt.tight_layout()
plt.show()

## 11. Final partition coloured from the photo

Each block's cells are rasterized in the **source** photo (full resolution, `H` scaled by `1/scale`), eroded by 0.12 cell edge to keep the rims, gaps and neighbours out, then converted to HSV:

* **hue**: circular mean of the hue angle, weighted by saturation (grey pixels carry no hue);
* **saturation**: median;
* **brightness**: mode of the V histogram (64 bins), so glare and shadow tails don't drag it.

The block is filled with that HSV colour.

In [ ]:
final_labels = regions(final_cut)
final_sizes = np.bincount(final_labels)
blank_regions = {region for region in range(len(final_sizes)) if blank_cells[final_labels == region].all()}
erode = np.ones((2 * round(0.12 * edge_px) + 1,) * 2, np.uint8)

block_colors = []
for region in range(len(final_sizes)):
	mask = np.zeros(source.shape[:2], np.uint8)
	for triangle in triangles[final_labels == region]:
		cv2.fillPoly(mask, [np.round(to_image(source_H, triangle) * 16).astype(np.int32)], 255, cv2.LINE_8, shift=4)
	pixels = source_hsv[cv2.erode(mask, erode) > 0].astype(np.float64)
	if not len(pixels):
		block_colors.append((0.5, 0.5, 0.5))
		print(f"region {region:>2}: {final_sizes[region]} cells, outside the photo")
		continue
	hue = circular_mean_hue(pixels) / 2
	saturation = np.median(pixels[:, 1])
	counts, edges = np.histogram(pixels[:, 2], bins=64, range=(0, 256))
	value = (edges[np.argmax(counts)] + edges[np.argmax(counts) + 1]) / 2
	rgb = cv2.cvtColor(np.uint8([[[round(hue) % 180, round(saturation), round(value)]]]), cv2.COLOR_HSV2RGB)[0, 0] / 255
	block_colors.append(rgb)
	print(f"{'blank' if region in blank_regions else 'block'} {region:>2}: {final_sizes[region]} cells, {len(pixels):>7} px, hue {hue * 2:6.1f}°, sat {saturation:5.1f}, value mode {value:5.1f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 11))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title("photo")
axes[1].imshow(np.full_like(image, 255))
for region, color in enumerate(block_colors):
	for triangle in triangles[final_labels == region]:
		if region in blank_regions:
			axes[1].add_patch(Polygon(to_image(H, triangle), closed=True, facecolor="white", edgecolor="gray", hatch="//", linewidth=0.5))
		else:
			axes[1].add_patch(Polygon(to_image(H, triangle), closed=True, facecolor=color, edgecolor=color, linewidth=0.5))
# Only cuts between two different blocks; a cut with the same block on both sides is a stray line inside a piece.
separating = final_cut & (final_labels[internal_pairs[:, 0]] != final_labels[internal_pairs[:, 1]])
print(f"{separating.sum()} separating cuts, {(final_cut & ~separating).sum()} stray cuts inside blocks (not drawn)")
axes[1].add_collection(LineCollection(projected_edges[separating], colors="black", linewidths=1.8))
axes[1].add_collection(LineCollection(to_image(H, outline.reshape(-1, 2)).reshape(-1, 2, 2), colors="black", linewidths=1.8))
axes[1].set_title(f"final partition: {len(final_sizes) - len(blank_regions)} blocks + {len(blank_regions)} blank, coloured by mean hue / median saturation / brightness mode")
for ax in axes:
	ax.set_xlim(0, image.shape[1])
	ax.set_ylim(image.shape[0], 0)
	ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Saved-board JSON for `hexiamond.vue`

The app persists its board in `localStorage["hexiamond.board.v1"]` as `{shapeId, placements: [{blockId, orientationId, translation}]}`, and on load re-resolves each entry against `buildShape(shapeId).placements` by exact `(orientationId, translation)`.
So the ids must come from the same enumeration as `geometry.ts`; it is ported verbatim below:

* `rotatePoint` / `reflectPoint` / `packPoint` / `normalize` on `[x, y, parity]` lattice points;
* `makeOrientations`: for each pattern, then its mirror, every `spin` rotation, deduplicated in first-seen order — the index is `orientationId`;
* `buildShape` placements: packed x offset in `[-14, width + 14)` step 2, y in `[-8, rows + 8)`, kept when every point is on the board; `translation = [x / 2, y]`.

Each recognized block's cell set is looked up among all placements; every 6-cell connected region is some hexiamond, and the 12 std blocks are the 12 distinct ones, so each region matches exactly one block.

In [ ]:
ROTATION = np.array([[1, 1], [-1, 0]])

def rotate_point(point, n):
	spin = n % 6
	parity = 1 - point[2] if spin % 2 else point[2]
	offset = ((spin + (1 if point[2] else 2)) // 3 % 2, (spin + (0 if point[2] else 1)) // 3 % 2)
	matrix = np.linalg.matrix_power(ROTATION, spin)
	return (int(point[0] * matrix[0][0] + point[1] * matrix[1][0] - offset[0]), int(point[0] * matrix[0][1] + point[1] * matrix[1][1] - offset[1]), parity)

def reflect_point(point):
	return (point[1] - point[0] - 1, point[1], 1) if point[2] else (point[1] - point[0], point[1], 0)

def pack_point(point):
	return (point[0] * 2 + point[2], point[1])

def normalize_points(points):
	shift_x = min(p[0] for p in points) // 2 * 2
	min_y = min(p[1] for p in points)
	return sorted(((p[0] - shift_x, p[1] - min_y) for p in points), key=lambda p: (p[1], p[0]))

def make_orientations(block):
	result = []
	def add(source):
		for spin in range(block["spin"]):
			points = normalize_points([pack_point(rotate_point(p, spin)) for p in source])
			if points not in result:
				result.append(points)
	for pattern in block["patterns"]:
		source = [tuple(p) for p in pattern]
		add(source)
		add([reflect_point(p) for p in source])
	return result

board_set = set(board_points)
width = max(len(row) for row in shape["board"])
placement_of = {}
placement_counts = {}
for block in shape["blocks"]:
	count = 0
	for orientation_id, points in enumerate(make_orientations(block)):
		for x in range(-14, width + 14, 2):
			for y in range(-8, len(shape["board"]) + 8):
				moved = frozenset((p[0] + x, p[1] + y) for p in points)
				if moved <= board_set:
					placement_of.setdefault(moved, []).append({"blockId": block["id"], "orientationId": orientation_id, "translation": [x // 2, y]})
					count += 1
	placement_counts[block["name"]] = count
print(f"{sum(placement_counts.values())} placements: " + ", ".join(f"{name} {count}" for name, count in placement_counts.items()))

blocks_by_id = {block["id"]: block for block in shape["blocks"]}
saved, region_block = [], {}
for region in range(len(final_sizes)):
	if region in blank_regions:
		print(f"region {region:>2}: blank ({final_sizes[region]} cells), no block")
		continue
	cells = frozenset(board_points[cell] for cell in np.flatnonzero(final_labels == region))
	matches = placement_of.get(cells, [])
	if len(matches) != 1:
		print(f"region {region}: {final_sizes[region]} cells, {len(matches)} matching placements - skipped")
		continue
	saved.append(matches[0])
	region_block[region] = matches[0]["blockId"]
	block = blocks_by_id[matches[0]["blockId"]]
	print(f"region {region:>2} -> {block['name']:<6} (id {block['id']:>2}) orientation {matches[0]['orientationId']:>2} translation {matches[0]['translation']}")
used = [entry["blockId"] for entry in saved]
duplicates = sorted({block_id for block_id in used if used.count(block_id) > 1})
print(f"{len(saved)} placements, {len(set(used))} distinct blocks" + (f"; DUPLICATE block ids {duplicates}" if duplicates else ""))

state = {"shapeId": SHAPE_ID, "placements": saved}
board_json = json.dumps(state)
out_path = IMAGE_PATH.parent / "lines" / f"{IMAGE_PATH.stem}.board.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(board_json + "\n")
print(f"\nwrote {out_path.relative_to(REPO)}\n\n{board_json}")
print(f"\nload in the app's devtools console, then reload:\nlocalStorage.setItem(\"hexiamond.board.v1\", '{board_json}')")

In [ ]:
# The saved placements drawn in the app's own block colours, beside the photo-coloured partition.
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
for ax, title in zip(axes, ("photo colours (§11)", "saved placements, app block colours + names")):
	ax.set_xlim(0, image.shape[1])
	ax.set_ylim(image.shape[0], 0)
	ax.set_aspect("equal")
	ax.axis("off")
	ax.set_title(title)
for region in range(len(final_sizes)):
	cells = triangles[final_labels == region]
	block = blocks_by_id.get(region_block.get(region))
	for triangle in cells:
		axes[0].add_patch(Polygon(to_image(H, triangle), closed=True, facecolor=block_colors[region], edgecolor=block_colors[region], linewidth=0.5))
		axes[1].add_patch(Polygon(to_image(H, triangle), closed=True, facecolor=block["color"] if block else "white", edgecolor="none", hatch=None if block else "//"))
	if block:
		axes[1].text(*to_image(H, cells.reshape(-1, 2).mean(axis=0, keepdims=True))[0], block["name"], ha="center", va="center", fontsize=11, weight="bold")
for ax in axes:
	ax.add_collection(LineCollection(projected_edges[separating], colors="black", linewidths=1.8))
	ax.add_collection(LineCollection(to_image(H, outline.reshape(-1, 2)).reshape(-1, 2, 2), colors="black", linewidths=1.8))
plt.tight_layout()
plt.show()

## Notes

* Single segments have ~1° angular precision (a 1px endpoint error on a 50px segment), so 0.1° modes are noisy; the means are steadier. The mode-based linear part is only a seed — the projective refinement in 6.3 re-estimates all of `a, b, c, d` anyway.
* Piece borders show up as two or three parallel lines (top rim, gap, shadow); the clipped loss tolerates this but it limits inliers to ~60%.
* The lattice term can't tell the board apart from a copy shifted by one cell; that is settled only by the outermost-segment seed. A wider frame or parallel clutter outside the board could put it one cell off — adding an outline term or the chosen board shape's extent would remove that ambiguity.